# Notebook 02 — Two baselines (fair comparison)

Notebook 01 already made: labels, splits, and the recipe book.

This notebook scores **two different methods** on the **same exams**:

| Method | Idea | Learns from train? |
|--------|------|--------------------|
| **A. Dictionary** | Look up warhead/E3 recipes and paint atoms | No (uses a book) |
| **B. XGBoost** | Learn which bonds to cut from train molecules | Yes |

**Fairness fixes in this notebook:**
1. Dictionary uses **train-only recipes** (closed book on unseen splits)
2. XGBoost atom score allows **W↔E3 swap** (don't punish name swaps)
3. Morgan SVD is fit on **train only** (no test leakage)
4. Bond ranking metric is named **`bond_pr_auc`** (not ROC)

Output: `outputs/baselines_metrics.json`


## 0. Paths


In [7]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)


ROOT = d:\Protac_gen\Yashi


## 1. Imports + shared labels

`0 = warhead`, `1 = linker`, `2 = E3`


In [8]:
from __future__ import annotations

import json
import pickle
from dataclasses import dataclass

import networkx as nx
import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
from sklearn.decomposition import TruncatedSVD
from xgboost import XGBClassifier

RDLogger.DisableLog("rdApp.*")

WARHEAD, LINKER, E3 = 0, 1, 2


## 2. Small chemistry helpers (used by both methods)

These are the same ideas as notebook 01, kept short so this notebook runs alone.


In [ ]:
@dataclass
class RefFragment:
    smiles: str
    mol: Chem.Mol
    n_atoms: int


def prepare_reference_library(smiles_list, min_atoms=5, max_atoms=60):
    """Clean SMILES into a recipe book (largest pieces first)."""
    refs = []
    for smi in smiles_list:
        if not isinstance(smi, str) or not smi:
            continue
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        n = m.GetNumHeavyAtoms()
        if min_atoms <= n <= max_atoms:
            refs.append(RefFragment(Chem.MolToSmiles(m), m, n))
    refs.sort(key=lambda r: r.n_atoms, reverse=True)
    return refs


def best_match(mol, refs, forbidden=None):
    """Find first recipe that fits; return (atom_set, recipe_smiles) or None."""
    forbidden = forbidden or set()
    for ref in refs:
        if ref.n_atoms > mol.GetNumAtoms():
            continue
        for match in mol.GetSubstructMatches(ref.mol, uniquify=True, useChirality=False):
            atoms = set(match)
            if atoms.isdisjoint(forbidden):
                return atoms, ref.smiles
    return None


def boundary_bonds(mol, labels):
    """Bonds that sit between two different part labels."""
    out = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if labels[i] != labels[j]:
            out.append(b.GetIdx())
    return out


def reassembles(mol, labels):
    """True if cutting at boundaries gives exactly 3 valid fragments."""
    cuts = boundary_bonds(mol, labels)
    if not cuts:
        return False, 1
    frag_mol = Chem.FragmentOnBonds(mol, cuts, addDummies=True)
    frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
    if len(frags) != 3:
        return False, len(frags)
    atoms_seen = 0
    for f in frags:
        try:
            Chem.SanitizeMol(f)
        except Exception:
            return False, len(frags)
        atoms_seen += sum(1 for a in f.GetAtoms() if a.GetAtomicNum() != 0)
    return atoms_seen == mol.GetNumAtoms(), 3


def atom_accuracy(true_labels, pred_labels, allow_we_swap=False):
    """Fraction of atoms with matching labels.

    If allow_we_swap=True, also try swapping warhead↔E3 and keep the better score.
    """ 
    n = len(true_labels)
    if n == 0:
        return 0.0

    def _acc(pred):
        return sum(a == b for a, b in zip(true_labels, pred)) / n

    best = _acc(pred_labels)
    if allow_we_swap:
        swapped = []
        for x in pred_labels:
            if x == WARHEAD:
                swapped.append(E3)
            elif x == E3:
                swapped.append(WARHEAD)
            else:
                swapped.append(LINKER)
        best = max(best, _acc(swapped))
    return best


def pr_auc(y_true, scores):
    """Average precision (area under precision-recall curve)."""
    if not y_true:
        return 0.0
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    n_pos = sum(y_true)
    if n_pos == 0:
        return 0.0
    tp = fp = 0
    ap = 0.0
    prev_recall = 0.0
    for i in order:
        if y_true[i]:
            tp += 1
        else:
            fp += 1
        prec = tp / (tp + fp)
        rec = tp / n_pos
        ap += prec * (rec - prev_recall)
        prev_recall = rec
    return ap


print("helpers ready")


helpers ready


## 3. Load notebook-01 outputs


In [10]:
records = [json.loads(line) for line in open(PROCESSED / "labeled_protacs.jsonl")]
splits = pickle.load(open(PROCESSED / "splits.pkl", "rb"))
refs_raw = pickle.load(open(PROCESSED / "refs.pkl", "rb"))

# Full recipe books (dictionary method will shrink these per-split)
all_wh_refs = prepare_reference_library(refs_raw["warheads"])
all_e3_refs = prepare_reference_library(refs_raw["e3"])

print("molecules:", len(records))
print("splits:", list(splits))
print("full warhead recipes:", len(all_wh_refs))
print("full E3 recipes:", len(all_e3_refs))


molecules: 4294
splits: ['random', 'unseen_warhead', 'unseen_e3', 'fingerprint_ood', 'newer_chemotype']
full warhead recipes: 1723
full E3 recipes: 184


---
# METHOD A — Dictionary baseline (closed book)

**Idea:** rematch warhead/E3 recipes on the test SMILES.

**Fair rule:** for each split, the book may only contain recipes that appear in that split's **train** molecules.

So on `unseen_warhead`, a rare warhead recipe is usually **missing** from the book → dictionary should fail more often.


### A1. Build a train-only recipe book


In [11]:
def train_only_library(records, train_idx, all_wh, all_e3):
    """Keep only recipes seen in TRAIN molecules of this split."""
    wh_seen = {records[i]["wh_ref"] for i in train_idx}
    e3_seen = {records[i]["e3_ref"] for i in train_idx}
    wh = [r for r in all_wh if r.smiles in wh_seen]
    e3 = [r for r in all_e3 if r.smiles in e3_seen]
    return wh, e3


# quick demo on unseen_warhead
_tr = splits["unseen_warhead"]["train"]
_wh, _e3 = train_only_library(records, _tr, all_wh_refs, all_e3_refs)
print("unseen_warhead train molecules:", len(_tr))
print("warhead recipes kept:", len(_wh), "/", len(all_wh_refs))
print("E3 recipes kept:", len(_e3), "/", len(all_e3_refs))


unseen_warhead train molecules: 3425
warhead recipes kept: 459 / 1723
E3 recipes kept: 63 / 184


### A2. Predict one molecule with a (possibly restricted) book


In [12]:
def dictionary_predict(smiles, wh_refs, e3_refs):
    """Paint atoms using recipe lookup. Return labels or None if match fails."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    if not (15 <= n <= 120):
        return None

    wh = best_match(mol, wh_refs)
    if wh is None:
        return None
    e3 = best_match(mol, e3_refs, forbidden=wh[0])
    if e3 is None:
        return None

    labels = [LINKER] * n
    for a in wh[0]:
        labels[a] = WARHEAD
    for a in e3[0]:
        labels[a] = E3
    if LINKER not in labels:
        return None
    return labels


print("dictionary_predict ready")


dictionary_predict ready


### A3. Score dictionary on one split's test set


In [13]:
def score_dictionary(records, train_idx, test_idx, all_wh, all_e3):
    """Train-only book + evaluate on test molecules."""
    wh_refs, e3_refs = train_only_library(records, train_idx, all_wh, all_e3)

    hits = exact3 = reasm = 0
    atom_correct = atom_tot = 0

    for idx in test_idx:
        rec = records[idx]
        true = rec["labels"]
        pred = dictionary_predict(rec["smiles"], wh_refs, e3_refs)

        if pred is None:
            # failed lookup: count all atoms in denominator, zero credit
            atom_tot += len(true)
            continue

        hits += 1
        atom_correct += sum(a == b for a, b in zip(true, pred))
        atom_tot += len(true)

        mol = Chem.MolFromSmiles(rec["smiles"])
        ok, nfrag = reassembles(mol, pred)
        exact3 += int(nfrag == 3)
        reasm += int(ok)

    n = max(len(test_idx), 1)
    return {
        "coverage": hits / n,
        "atom_acc": atom_correct / max(atom_tot, 1),
        "exact_3_frag": exact3 / n,
        "reassembly": reasm / n,
        "n_test_molecules": len(test_idx),
        "n_wh_recipes_used": len(wh_refs),
        "n_e3_recipes_used": len(e3_refs),
    }


print("score_dictionary ready")


score_dictionary ready


### A4. Run dictionary on every split


In [14]:
print("=== METHOD A: Dictionary (train-only book) ===")
dict_metrics = {}

for sname, s in splits.items():
    m = score_dictionary(records, s["train"], s["test"], all_wh_refs, all_e3_refs)
    dict_metrics[sname] = m
    print(
        f"  {sname:18s}  "
        f"cover={m['coverage']:.3f}  "
        f"atom={m['atom_acc']:.3f}  "
        f"exact3={m['exact_3_frag']:.3f}  "
        f"reasm={m['reassembly']:.3f}  "
        f"(book W={m['n_wh_recipes_used']}, E3={m['n_e3_recipes_used']})"
    )


=== METHOD A: Dictionary (train-only book) ===
  random              cover=0.984  atom=0.955  exact3=0.981  reasm=0.981  (book W=822, E3=77)
  unseen_warhead      cover=0.062  atom=0.057  exact3=0.000  reasm=0.000  (book W=459, E3=63)
  unseen_e3           cover=0.329  atom=0.309  exact3=0.000  reasm=0.000  (book W=732, E3=5)
  fingerprint_ood     cover=0.932  atom=0.891  exact3=0.930  reasm=0.930  (book W=808, E3=72)
  newer_chemotype     cover=0.485  atom=0.470  exact3=0.464  reasm=0.464  (book W=759, E3=72)


---
# METHOD B — XGBoost bond-cut baseline 

**Idea:**
1. Each bond gets a feature row (local chemistry + molecule context)
2. Train learns: is this bond a cut bond?
3. On test: pick top-2 cut bonds → split into 3 pieces → paint W/L/E3

**Fair rules:**
- Morgan SVD fit on **train only**
- Atom accuracy uses best of normal / **W↔E3 swapped** labels


### B1. Bond features + bridge score


In [15]:
def bridge_scores(mol):
    """Simple importance score for bridge bonds (0 for non-bridges)."""
    G = nx.Graph()
    for a in mol.GetAtoms():
        G.add_node(a.GetIdx())
    for b in mol.GetBonds():
        G.add_edge(b.GetBeginAtomIdx(), b.GetEndAtomIdx())

    n = G.number_of_nodes()
    out = {}
    if n < 2:
        return out
    denom = n * (n - 1) / 2
    bridges = set(nx.bridges(G))

    for b in mol.GetBonds():
        u, v = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if (u, v) not in bridges and (v, u) not in bridges:
            continue
        H = G.copy()
        H.remove_edge(u, v)
        comps = list(nx.connected_components(H))
        if len(comps) == 2:
            a, bsz = len(comps[0]), len(comps[1])
            out[b.GetIdx()] = (a * bsz) / denom
    return out


def bond_features(mol, bond, bc, morgan_vec):
    """One numeric row for one bond = local flags + molecule context."""
    i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    a1, a2 = mol.GetAtomWithIdx(i), mol.GetAtomWithIdx(j)

    is_ring = int(bond.IsInRing())
    sp3 = 0.5 * (
        int(a1.GetHybridization() == Chem.HybridizationType.SP3)
        + int(a2.GetHybridization() == Chem.HybridizationType.SP3)
    )
    rotatable = int(
        (not is_ring)
        and bond.GetBondType() == Chem.BondType.SINGLE
        and a1.GetDegree() > 1
        and a2.GetDegree() > 1
    )
    bc_val = bc.get(bond.GetIdx(), 0.0)

    def _has_carbonyl_o(carbon):
        for nb in carbon.GetNeighbors():
            if nb.GetAtomicNum() != 8:
                continue
            bb = mol.GetBondBetweenAtoms(carbon.GetIdx(), nb.GetIdx())
            if bb is not None and bb.GetBondType() == Chem.BondType.DOUBLE:
                return True
        return False

    is_amide = int(
        (a1.GetAtomicNum() == 7 and a2.GetAtomicNum() == 6 and _has_carbonyl_o(a2))
        or (a2.GetAtomicNum() == 7 and a1.GetAtomicNum() == 6 and _has_carbonyl_o(a1))
    )
    is_ether = int(a1.GetAtomicNum() == 8 or a2.GetAtomicNum() == 8)

    bond_type = {
        Chem.BondType.SINGLE: 1.0,
        Chem.BondType.DOUBLE: 2.0,
        Chem.BondType.TRIPLE: 3.0,
        Chem.BondType.AROMATIC: 1.5,
    }.get(bond.GetBondType(), 1.0)

    local = np.array([
        is_ring, sp3, rotatable, bc_val,
        a1.GetAtomicNum(), a2.GetAtomicNum(), bond_type,
        is_amide, is_ether,
        int(a1.IsInRing()), int(a2.IsInRing()),
        int(a1.GetIsAromatic()), int(a2.GetIsAromatic()),
        int(bond.GetIsAromatic()),
        a1.GetDegree(), a2.GetDegree(),
        int(a1.GetFormalCharge()), int(a2.GetFormalCharge()),
    ], dtype=np.float32)
    return np.concatenate([local, morgan_vec])


print("bond feature helpers ready")


bond feature helpers ready


### B2. Morgan fingerprints + train-only SVD (no test leakage)


In [16]:
def morgan_matrix(records):
    """Raw Morgan bit matrix for all molecules (no SVD yet)."""
    fps = np.zeros((len(records), 256), dtype=np.float32)
    for i, rec in enumerate(records):
        mol = Chem.MolFromSmiles(rec["smiles"])
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=256)
        arr = np.zeros(256, dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fps[i] = arr.astype(np.float32)
    return fps


def morgan_svd_for_split(fps, train_idx, n_components=32):
    """Fit SVD on TRAIN rows only, then transform ALL rows with that fit."""
    n_comp = min(n_components, fps.shape[1] - 1, max(len(train_idx) - 1, 1))
    svd = TruncatedSVD(n_components=n_comp, random_state=0)
    svd.fit(fps[train_idx])          # fit on train only  <-- FIX 3
    return svd.transform(fps)


FPS = morgan_matrix(records)
print("Morgan matrix:", FPS.shape)


Morgan matrix: (4294, 256)


### B3. Build X (bonds × features) and y (cut / not-cut)


In [17]:
def build_bond_table(records, indices, morgan_svd_all):
    """Return X, y, and which molecule each bond-row belongs to."""
    X_rows, y_rows, group = [], [], []
    for idx in indices:
        rec = records[idx]
        mol = Chem.MolFromSmiles(rec["smiles"])
        bc = bridge_scores(mol)
        cut_set = set(boundary_bonds(mol, rec["labels"]))
        mvec = morgan_svd_all[idx]
        for b in mol.GetBonds():
            X_rows.append(bond_features(mol, b, bc, mvec))
            y_rows.append(1 if b.GetIdx() in cut_set else 0)
            group.append(idx)
    return np.stack(X_rows), np.array(y_rows), group


print("build_bond_table ready")


build_bond_table ready


### B4. Convert top-2 cuts → W / L / E3 atom labels


In [18]:
def cuts_to_labels(mol, cut_bond_indices):
    """Cut 2 bonds → 3 fragments → middle=linker, ends=W/E3 (size order)."""
    n = mol.GetNumAtoms()
    if not cut_bond_indices:
        return [LINKER] * n

    rw = Chem.RWMol(mol)
    for bidx in cut_bond_indices:
        try:
            bond = mol.GetBondWithIdx(bidx)
            rw.RemoveBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        except Exception:
            pass

    frags = Chem.GetMolFrags(rw.GetMol(), asMols=False)
    if len(frags) != 3:
        return [LINKER] * n

    frag_sets = [set(f) for f in frags]
    cut_pairs = []
    for bidx in cut_bond_indices:
        bond = mol.GetBondWithIdx(bidx)
        cut_pairs.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))

    # linker = fragment that touches both cut bonds
    linker_idx = -1
    for k, s in enumerate(frag_sets):
        if all(any(a in s for a in pair) for pair in cut_pairs):
            linker_idx = k
            break
    if linker_idx < 0:
        linker_idx = min(range(3), key=lambda k: len(frag_sets[k]))

    ends = [k for k in range(3) if k != linker_idx]
    ends.sort(key=lambda k: -len(frag_sets[k]))  # larger end -> warhead guess

    labels = [LINKER] * n
    for a in frag_sets[ends[0]]:
        labels[a] = WARHEAD
    for a in frag_sets[ends[1]]:
        labels[a] = E3
    return labels


print("cuts_to_labels ready")


cuts_to_labels ready


### B5. Train + score XGBoost on one split


In [19]:
def score_xgboost(records, train_idx, test_idx, fps):
    """Train on train bonds, predict top-2 cuts on test, score with W/E3 swap."""
    # FIX 3: SVD fit on train only
    morgan_svd = morgan_svd_for_split(fps, train_idx)

    X_tr, y_tr, _ = build_bond_table(records, train_idx, morgan_svd)
    X_te, y_te, grp_te = build_bond_table(records, test_idx, morgan_svd)

    pos = max(int(y_tr.sum()), 1)
    neg = max(int((y_tr == 0).sum()), 1)
    clf = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        tree_method="hist",
        scale_pos_weight=float(neg) / pos,
        n_jobs=-1,
        verbosity=0,
        eval_metric="logloss",
        random_state=0,
    )
    clf.fit(X_tr, y_tr)
    scores = clf.predict_proba(X_te)[:, 1]

    # group bond rows by molecule for fast top-2 selection
    rows_by_mol = {}
    for row_i, mol_i in enumerate(grp_te):
        rows_by_mol.setdefault(mol_i, []).append(row_i)

    exact3 = reasm = 0
    atom_sum = 0.0

    for idx in test_idx:
        rec = records[idx]
        mol = Chem.MolFromSmiles(rec["smiles"])
        row_ids = rows_by_mol[idx]
        ranked = sorted(row_ids, key=lambda r: -scores[r])
        first = row_ids[0]
        top2_local = [r - first for r in ranked[:2]]

        pred = cuts_to_labels(mol, top2_local)
        # FIX 2: best of normal / W↔E3 swap
        atom_sum += atom_accuracy(rec["labels"], pred, allow_we_swap=True)

        ok, nfrag = reassembles(mol, pred)
        exact3 += int(nfrag == 3)
        reasm += int(ok)

    n = max(len(test_idx), 1)
    return {
        "atom_acc": atom_sum / n,
        "exact_3_frag": exact3 / n,
        "reassembly": reasm / n,
        "n_test_molecules": len(test_idx),
        "bond_pr_auc": pr_auc(y_te.tolist(), scores.tolist()),  # FIX 4
    }


print("score_xgboost ready")


score_xgboost ready


### B6. Run XGBoost on every split


In [20]:
print("=== METHOD B: XGBoost (train-only SVD, W/E3-swap scoring) ===")
xgb_metrics = {}

for sname, s in splits.items():
    print("---", sname, "---")
    m = score_xgboost(records, s["train"], s["test"], FPS)
    xgb_metrics[sname] = m
    print(
        f"  atom={m['atom_acc']:.3f}  "
        f"exact3={m['exact_3_frag']:.3f}  "
        f"reasm={m['reassembly']:.3f}  "
        f"bond_pr_auc={m['bond_pr_auc']:.3f}"
    )


=== METHOD B: XGBoost (train-only SVD, W/E3-swap scoring) ===
--- random ---
  atom=0.840  exact3=0.853  reasm=0.853  bond_pr_auc=0.750
--- unseen_warhead ---
  atom=0.828  exact3=0.913  reasm=0.913  bond_pr_auc=0.255
--- unseen_e3 ---
  atom=0.839  exact3=0.933  reasm=0.933  bond_pr_auc=0.269
--- fingerprint_ood ---
  atom=0.816  exact3=0.846  reasm=0.846  bond_pr_auc=0.535
--- newer_chemotype ---
  atom=0.666  exact3=0.690  reasm=0.690  bond_pr_auc=0.472


---
# Compare + save

Dictionary and XGBoost are kept as **separate** result blocks.


In [21]:
print("\n======== SIDE-BY-SIDE (atom_acc) ========")
print(f"{'split':18s}  {'dictionary':>12s}  {'xgboost':>12s}")
for sname in splits:
    d = dict_metrics[sname]["atom_acc"]
    x = xgb_metrics[sname]["atom_acc"]
    print(f"{sname:18s}  {d:12.3f}  {x:12.3f}")

print("\nDictionary coverage (should drop on hard unseen splits):")
for sname in splits:
    print(f"  {sname:18s}  cover={dict_metrics[sname]['coverage']:.3f}")



======== SIDE-BY-SIDE (atom_acc) ========
split                 dictionary       xgboost
random                     0.955         0.840
unseen_warhead             0.057         0.828
unseen_e3                  0.309         0.839
fingerprint_ood            0.891         0.816
newer_chemotype            0.470         0.666

Dictionary coverage (should drop on hard unseen splits):
  random              cover=0.984
  unseen_warhead      cover=0.062
  unseen_e3           cover=0.329
  fingerprint_ood     cover=0.932
  newer_chemotype     cover=0.485


In [22]:
out = {
    "dictionary": dict_metrics,
    "xgboost": xgb_metrics,
    "notes": {
        "dictionary": "train-only recipe book per split",
        "xgboost": "train-only Morgan SVD; atom_acc allows W/E3 swap; bond_pr_auc is average precision",
    },
}

with open(OUT / "baselines_metrics.json", "w") as f:
    json.dump(out, f, indent=2)

print("saved", OUT / "baselines_metrics.json")
print("DONE — Notebook 02 complete.")


saved d:\Protac_gen\Yashi\outputs\baselines_metrics.json
DONE — Notebook 02 complete.
